# WTA Match Predictor — EDA y Feature Engineering

Pipeline completo: carga de datos, exploración, limpieza, clasificación de torneos, cálculo de features derivadas y generación del dataset de entrenamiento (`historico_partidos.csv`).

**Salidas de este notebook:**
- `src/data/wta_limpio.csv` — dataset WTA limpio con ELO precalculado
- `src/data/historico_partidos.csv` — dataset de features listo para entrenar

## 1. Imports y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from ydata_profiling import ProfileReport

In [ ]:
# Añadir utils al path para importar features.py desde cualquier entorno
sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'utils'))

In [ ]:
# Rutas relativas al repositorio (ejecutar desde src/notebooks/)
DATA_DIR  = os.path.join('..', 'data')
MODEL_DIR = os.path.join('..', 'model')

In [ ]:
#!pip install kagglehub

In [ ]:
# Descarga del dataset desde Kaggle (requiere autenticación con kagglehub)
# Dataset: https://www.kaggle.com/datasets/dissfya/wta-tennis-2007-2023-daily-update
# Alternativamente los archivos de estadísticas históricas están disponibles en:
# https://github.com/JeffSackmann/tennis_wta
import kagglehub

path = kagglehub.dataset_download("dissfya/wta-tennis-2007-2023-daily-update")
print("Path to dataset files:", path)

In [ ]:
# Si ya tienes wta.csv en data/, puedes cargarlo directamente:
# wta = pd.read_csv(os.path.join(DATA_DIR, 'wta.csv'), low_memory=False)

file_path = os.path.join(path, "wta.csv")
wta = pd.read_csv(file_path, low_memory=False)
print(f'Partidos cargados: {len(wta)}')
wta.tail()

## 2. Exploración inicial

In [ ]:
wta.info()

In [ ]:
# Profiling completo — genera wta_report.html con distribuciones, correlaciones y alertas
profile = ProfileReport(wta, title='WTA Profiling Report')
profile.to_file('wta_report.html')

## 3. Limpieza y transformaciones

In [ ]:
# Eliminar columnas no útiles para el modelo
# Court: muy desbalanceado (40k outdoor vs 400 indoor), la superficie ya lo captura
# Best of: en WTA siempre es al mejor de 3 sets
wta = wta.drop(columns=['Court', 'Best of'])

In [ ]:
# Unificar superficies: Greenset y Carpet tienen muy pocas entradas y son pistas rápidas → Hard
wta['Surface'] = wta['Surface'].replace({'Greenset': 'Hard', 'Carpet': 'Hard'})
print(wta['Surface'].value_counts())

In [ ]:
# Clasificar torneos por categoría usando palabras clave
# Los nombres cambian con patrocinadores pero la ciudad/torneo es estable

def clasificar_torneo(nombre):
    n = nombre.lower()
    
    if any(x in n for x in ['australian open', 'roland garros', 'french open',
                              'wimbledon', 'us open']):
        return 'GS'
    
    if any(x in n for x in ['wta finals', 'wta elite trophy', 'championships',
                              'tournament of champions', 'riyadh finals', 'wta tour championships']):
        return 'WTA_Finals'
    
    if any(x in n for x in ['indian wells', 'miami', 'madrid', 'roma', 'rome',
                              'internazionali', 'canada', 'toronto', 'rogers',
                              'canadian', 'cincinnati', 'beijing', 'china open',
                              'wuhan', 'doha', 'qatar', 'dubai', 'bnp paribas open',
                              'national bank open', 'sony ericsson open',
                              'western & southern financial group', 'shenzhen']):
        return 'WTA1000'
    
    if any(x in n for x in ['abu dhabi', 'abu dabi', 'adelaide', 'berlin', 'bett1open',
                              'eastbourne', 'rothesay', 'bad homburg', 'san jose',
                              'silicon valley', 'guangzhou', 'osaka', 'tokyo',
                              'toray pan pacific', 'seoul', 'strasbourg', 'birmingham',
                              'nottingham', 'brisbane', 'linz', 'merida', 'monterrey',
                              'charleston', 'stuttgart', 'porsche', 'washington',
                              'mubadala', 'guadalajara', 'gdl open akron',
                              'family circle cup', 'aegon', 'kremlin', 'pilot pen',
                              'new haven', 'bank of the west', 'sydney international',
                              'stanford', 'luxembourg', 'belgium', 'diamond games',
                              'fortis', 'san diego', 'zhengzhou', 'german']):
        return 'WTA500'
    
    return 'WTA250_o_menor'

wta['tournament_type'] = wta['Tournament'].map(clasificar_torneo)
print(wta['tournament_type'].value_counts())

In [ ]:
# Comprobación: torneos sin clasificar
sin_clasificar = wta[wta['tournament_type'] == 'WTA250_o_menor']['Tournament'].nunique()
print(f'Torneos únicos sin clasificar: {sin_clasificar}')

In [ ]:
# Eliminar Tournament (ya tenemos tournament_type) y convertir tipos
wta = wta.drop(columns='Tournament')

wta['Date']  = pd.to_datetime(wta['Date'], errors='coerce')
wta['Odd_1'] = pd.to_numeric(wta['Odd_1'], errors='coerce')
wta['Odd_2'] = pd.to_numeric(wta['Odd_2'], errors='coerce')

print('Nulos tras conversión:')
print(wta[['Date', 'Odd_1', 'Odd_2']].isna().sum())

In [ ]:
# Eliminar partidos sin fecha
wta = wta.dropna(subset=['Date'])

In [ ]:
# Limpiar odds: valores < 1 son códigos de sin datos (equivalente a -1)
# Los convertimos a NaN — se usarán para comparar con casas pero no para entrenar
wta.loc[wta['Odd_1'] < 1, 'Odd_1'] = None
wta.loc[wta['Odd_2'] < 1, 'Odd_2'] = None

In [ ]:
# Calcular probabilidades implícitas de las casas de apuestas
# Las odds indican cuánto paga cada euro apostado (ej: 2.5x). Las convierto a probabilidades:
# prob = 1/odd, luego normalizo para que sumen 1 (elimina el margen de la casa)
wta['prob_1'] = 1 / wta['Odd_1']
wta['prob_2'] = 1 / wta['Odd_2']
total = wta['prob_1'] + wta['prob_2']
wta['prob_1'] = wta['prob_1'] / total
wta['prob_2'] = wta['prob_2'] / total

print(wta['prob_1'].describe())

In [ ]:
# Convertir tipos de columnas categóricas y de texto
wta['Surface']  = wta['Surface'].astype(str)
wta['Round']    = wta['Round'].astype(str)
wta['Score']    = wta['Score'].astype(str)
wta['Player_1'] = wta['Player_1'].astype(str)
wta['Player_2'] = wta['Player_2'].astype(str)
wta['Winner']   = wta['Winner'].astype(str)

In [ ]:
# Crear target: 1 si gana Player_1, 0 si gana Player_2
def asignar_target(fila):
    if fila['Winner'] == fila['Player_1']:
        return 1
    else:
        return 0

wta['target'] = wta.apply(asignar_target, axis=1)
print(wta['target'].value_counts())

In [ ]:
# Guardar wta limpio (sin ELO aún) — se sobreescribirá en la sección 4 con ELO incluido
wta.to_csv(os.path.join(DATA_DIR, 'wta_limpio.csv'), index=False)
print('wta_limpio.csv guardado')

In [ ]:
wta = pd.read_csv(os.path.join(DATA_DIR, 'wta_limpio.csv'), parse_dates=['Date'])

## 4. Feature Engineering

Calculamos features derivadas para cada partido usando solo información disponible **antes** de ese partido (corte temporal estricto para evitar data leakage).

Las funciones auxiliares (`forma_reciente`, `winrate`, `headtohead`, etc.) están también en `src/utils/features.py` para su uso en la app y en la simulación.

### 4.1 ELO — cálculo previo a las features de partido

El ELO se precalcula en orden cronológico sobre todo el dataset antes de construir las features de partido, porque cada partido necesita el ELO acumulado hasta ese momento. Se calculan dos variantes:
- **ELO por superficie**: captura el rendimiento específico en cada tipo de pista.
- **ELO global**: rendimiento general independiente de superficie.

In [ ]:
def calcular_elo_global(df, k=32, elo_inicial=1500):
    """
    Precalcula ELO global (independiente de superficie) para cada partido.
    Añade elo_global_p1, elo_global_p2, elo_global_diff al dataframe.
    
    IMPORTANTE: df debe estar ordenado por fecha ascendente antes de llamar esto.
    El ELO de cada partido = ELO ANTES de jugarlo (sin leakage).
    """
    elo_ratings = {}  # {jugadora: elo}

    def get_elo(jugadora):
        return elo_ratings.get(jugadora, elo_inicial)

    def expected(elo_a, elo_b):
        return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

    elo_p1_list, elo_p2_list = [], []

    for _, row in df.iterrows():
        p1        = row['Player_1']
        p2        = row['Player_2']
        resultado = row['target']

        elo_p1 = get_elo(p1)
        elo_p2 = get_elo(p2)

        elo_p1_list.append(elo_p1)
        elo_p2_list.append(elo_p2)

        exp_p1 = expected(elo_p1, elo_p2)

        elo_ratings[p1] = elo_p1 + k * (resultado - exp_p1)
        elo_ratings[p2] = elo_p2 + k * ((1 - resultado) - (1 - exp_p1))

    df = df.copy()
    df['elo_global_p1']   = elo_p1_list
    df['elo_global_p2']   = elo_p2_list
    df['elo_global_diff'] = df['elo_global_p1'] - df['elo_global_p2']

    return df

In [ ]:
def calcular_elo_superficie(df, k=32, elo_inicial=1500):
    """
    Precalcula ELO por superficie para cada partido.
    Añade columnas elo_p1, elo_p2, elo_diff al dataframe.
    
    IMPORTANTE: df debe estar ordenado por fecha ascendente antes de llamar esto.
    El ELO de cada partido = ELO ANTES de jugarlo (sin leakage).
    """
    elo_ratings = {}  # {jugadora: {superficie: elo}}

    def get_elo(jugadora, superficie):
        return elo_ratings.get(jugadora, {}).get(superficie, elo_inicial)

    def update_elo(jugadora, superficie, nuevo_valor):
        if jugadora not in elo_ratings:
            elo_ratings[jugadora] = {}
        elo_ratings[jugadora][superficie] = nuevo_valor

    def expected(elo_a, elo_b):
        return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

    elo_p1_list, elo_p2_list = [], []

    for _, row in df.iterrows():
        p1         = row['Player_1']
        p2         = row['Player_2']
        superficie = row['Surface']
        resultado  = row['target']  # 1 = gana p1, 0 = gana p2

        elo_p1 = get_elo(p1, superficie)
        elo_p2 = get_elo(p2, superficie)

        # Guardamos ANTES de actualizar → sin leakage
        elo_p1_list.append(elo_p1)
        elo_p2_list.append(elo_p2)

        exp_p1 = expected(elo_p1, elo_p2)
        exp_p2 = 1 - exp_p1

        update_elo(p1, superficie, elo_p1 + k * (resultado - exp_p1))
        update_elo(p2, superficie, elo_p2 + k * ((1 - resultado) - exp_p2))

    df = df.copy()
    df['elo_p1']   = elo_p1_list
    df['elo_p2']   = elo_p2_list
    df['elo_diff'] = df['elo_p1'] - df['elo_p2']

    return df

In [ ]:
wta = calcular_elo_superficie(wta)
wta = calcular_elo_global(wta)

# Sobreescribir wta_limpio con ELO incluido — necesario para la simulación del torneo
wta.to_csv(os.path.join(DATA_DIR, 'wta_limpio.csv'), index=False)
print('wta_limpio.csv guardado (con ELO)')

### 4.2 Funciones de features por partido

In [ ]:
def forma_reciente(df, jugadora, fecha_limite, meses=2):
    """Win rate de una jugadora en los N meses anteriores a la fecha.
    Devuelve 0 si no tiene partidos (lesión o pausa larga)"""
    fecha_inicio = fecha_limite - pd.DateOffset(months=meses)
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] < fecha_limite) &
        (df['Date'] >= fecha_inicio)
    )
    partidos = df[mask]
    if len(partidos) == 0:
        return 0
    victorias = (partidos['Winner'] == jugadora).sum()
    return victorias/len(partidos)

In [ ]:
def winrate(df, jugadora, fecha_limite, superficie=None, ronda=None):
    """Win rate histórico de una jugadora, filtrable por superficie y/o ronda.
    Devuelve 0.4 si no tiene historial (ligeramente por debajo de neutro = novata en esa condición)"""
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] < fecha_limite)
    )
    if superficie:
        mask &= (df['Surface'] == superficie)
    if ronda:
        mask &= (df['Round'] == ronda)
    partidos = df[mask]
    if len(partidos) == 0:
        return 0.4
    victorias = (partidos['Winner'] == jugadora).sum()
    return victorias/len(partidos)

In [ ]:
def headtohead(df, p1, p2, fecha_limite):
    """% de victorias de p1 sobre p2 en enfrentamientos directos previos a la fecha.
    Devuelve 0.5 si nunca se han enfrentado (neutro)"""
    mask = (
        (
            ((df['Player_1'] == p1) & (df['Player_2'] == p2)) |
            ((df['Player_1'] == p2) & (df['Player_2'] == p1))
        ) & (df['Date'] < fecha_limite)
    )
    partidos = df[mask]
    if len(partidos) == 0:
        return 0.5
    victorias_p1 = (partidos['Winner'] == p1).sum()
    return victorias_p1/len(partidos)

In [ ]:
def experiencia(df, jugadora, fecha_limite):
    """Número total de partidos jugados por la jugadora antes de la fecha"""
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] < fecha_limite)
    )
    return df[mask].shape[0]

In [ ]:
def inactividad(df, jugadora, fecha_limite):
    """Días desde el último partido de la jugadora antes de la fecha.
    Devuelve 1000 si no tiene historial (sin datos previos)"""
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] < fecha_limite)
    )
    partidos = df[mask].sort_values('Date', ascending=False)
    if len(partidos) == 0:
        return 1000
    ultimo = partidos.iloc[0]['Date']
    dias = (fecha_limite - ultimo).days
    return dias

In [ ]:
def tendencia_ranking(df, jugadora, fecha_limite, meses=6):
    """Diferencia de ranking entre hace N meses y la fecha límite.
    Positivo = mejora (bajó en el ranking numérico). Devuelve 0 si no hay datos."""
    fecha_inicio = fecha_limite - pd.DateOffset(months=meses)
    
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] > fecha_inicio) &
        (df['Date'] <= fecha_limite)
    )
    
    partidos = df[mask].sort_values('Date', ascending=True)
    
    if len(partidos) == 0:
        return 0
    
    primero = partidos.iloc[0]
    ultimo  = partidos.iloc[-1]
    
    def get_ranking(partido, jugadora):
        if partido['Player_1'] == jugadora:
            return partido['Rank_1']
        else:
            return partido['Rank_2']
    
    ranking_antes = get_ranking(primero, jugadora)
    ranking_ahora = get_ranking(ultimo, jugadora)
    
    if pd.isna(ranking_antes) or pd.isna(ranking_ahora) or ranking_antes == 0 or ranking_ahora == 0:
        return None
    
    return ranking_antes - ranking_ahora

### 4.3 Comprobación de funciones

Verificación rápida de que las funciones devuelven valores razonables antes de ejecutar el bucle completo.

In [ ]:
jugadora      = 'Badosa P.'
fecha_limite  = pd.to_datetime('2026-01-01')

print(f"Experiencia:      {experiencia(wta, jugadora, fecha_limite)} partidos")
print(f"H2H vs Sabalenka: {headtohead(wta, jugadora, 'Sabalenka A.', fecha_limite):.2%}")
print(f"Win rate Clay:    {winrate(wta, jugadora, fecha_limite, superficie='Clay'):.2%}")
print(f"Forma reciente:   {forma_reciente(wta, jugadora, fecha_limite):.2%}")
print(f"Tendencia ranking:{tendencia_ranking(wta, jugadora, fecha_limite)}")
print(f"Días inactiva:    {inactividad(wta, jugadora, fecha_limite)}")

### 4.4 Construcción del dataset de features

⚠️ **Este bucle tarda ~30 minutos.** El resultado se guarda en CSV (`historico_partidos.csv`) para no tener que repetirlo.

Nota: `inactividad` y `tendencia_ranking` se calcularon y evaluaron no se incluyeron finalmente del modelo (ver sección de análisis de variables).

In [ ]:
# Reset index para usar como match_id
wta = wta.reset_index()

features = []
for i, partido in wta.iterrows():
    p1    = partido['Player_1']
    p2    = partido['Player_2']
    fecha = partido['Date']
    
    row = {
        # Identificador y metadata
        'match_id': i,
        'date':     partido['Date'],
        'odd_1':    partido['prob_1'],
        'odd_2':    partido['prob_2'],

        # Features del partido
        'surface':         partido['Surface'],
        'round':           partido['Round'],
        'tournament_type': partido['tournament_type'],

        # Features calculadas (solo info anterior al partido — sin leakage)
        'rank_diff':           float(partido['Rank_1']) - float(partido['Rank_2']),
        'wins2meses_p1':       forma_reciente(wta, p1, fecha),
        'wins2meses_p2':       forma_reciente(wta, p2, fecha),
        'ratio_superficie_p1': winrate(wta, p1, fecha, superficie=partido['Surface']),
        'ratio_superficie_p2': winrate(wta, p2, fecha, superficie=partido['Surface']),
        'h2h':                 headtohead(wta, p1, p2, fecha),
        'ratio_ronda_p1':      winrate(wta, p1, fecha, ronda=partido['Round']),
        'ratio_ronda_p2':      winrate(wta, p2, fecha, ronda=partido['Round']),
        'experiencia_p1':      experiencia(wta, p1, fecha),
        'experiencia_p2':      experiencia(wta, p2, fecha),
        # Evaluadas y descartadas — no mejoraban el modelo:
        # 'inactividad_p1':   inactividad(wta, p1, fecha),
        # 'inactividad_p2':   inactividad(wta, p2, fecha),
        # ELO precalculado (leído del df, no recalculado aquí)
        'elo_p1':          partido['elo_p1'],
        'elo_p2':          partido['elo_p2'],
        'elo_diff':        partido['elo_diff'],
        'elo_global_p1':   partido['elo_global_p1'],
        'elo_global_p2':   partido['elo_global_p2'],
        'elo_global_diff': partido['elo_global_diff'],

        # Target
        'target': partido['target']
    }
    features.append(row)

historico_partidos = pd.DataFrame(features)

# Feature de inexperiencia — flag binario para jugadoras con menos de 10 partidos registrados
historico_partidos['is_new_p1'] = (historico_partidos['experiencia_p1'] < 10).astype(int)
historico_partidos['is_new_p2'] = (historico_partidos['experiencia_p2'] < 10).astype(int)

print(f'Dataset construido: {len(historico_partidos)} partidos')
print(historico_partidos.info())

## 5. Limpieza del dataset de features

In [ ]:
# Eliminar partidos de 2007: primer año del dataset, las features derivadas están muy incompletas
# porque no hay historial previo para calcularlas
historico_partidos = historico_partidos[historico_partidos['date'].dt.year > 2007]
print(f'Partidos tras eliminar 2007: {len(historico_partidos)}')

In [ ]:
# Verificar nulos
print(historico_partidos.isnull().sum())

In [ ]:
# Guardar — carga posterior: pd.read_csv('historico_partidos.csv', parse_dates=['date'])
historico_partidos.to_csv(os.path.join(DATA_DIR, 'historico_partidos.csv'), index=False)
print('✓ historico_partidos.csv guardado')

## 6. Exploración: Clustering de perfiles de jugadoras

> **Nota:** Esta sección es una exploración adicional que se realizó para intentar caracterizar perfiles de juego mediante clustering no supervisado. Los datos provienen de un segundo dataset con estadísticas (especialmente de saque) de partido, disponible en [JeffSackmann/tennis_wta](https://github.com/JeffSackmann/tennis_wta) (archivos `wta_matches_YYYY.csv`).
>
> **Conclusión adelantada:** el clustering no produjo grupos interpretables y se descartó como feature del modelo. Se documenta aquí el proceso completo por trazabilidad.

In [ ]:
import glob

# Datasets de estadísticas detalladas de partido (Jeff Sackmann / tennis_wta en GitHub). Un dataset por año. Se utilizaron por coherencia
# los de los años desde 2007 a 2026. 
# Nota. Este dataset no se usó desde el inicio porque 
# Descarga manual: https://github.com/JeffSackmann/tennis_wta
# Rutas locales — ajusta a tu entorno:
# STATS_DIR = r'C:\Users\NaiaJon\Documents\Naia\BootcampDataScience\Datos ML\stats'

import os
print(os.listdir(STATS_DIR))

In [ ]:
# Cargar todos los archivos de estadísticas anuales
archivos = glob.glob(os.path.join(STATS_DIR, '*.csv'))

dfs = []
for f in archivos:
    dfs.append(pd.read_csv(f))
df_stats = pd.concat(dfs, ignore_index=True)
print(df_stats.shape)

In [ ]:
df_stats.tail()

In [ ]:
df_stats.columns

In [ ]:
# Selección de columnas de saque relevantes
df = df_stats[['winner_name', 'loser_name', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon',
       'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt',
       'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced']]

df.info()

In [ ]:
# l_SvGms / w_SvGms: número de juegos de servicio ganados — solo 27k non-null de ~40k. Se eliminan
df = df.drop(['l_SvGms', 'w_SvGms'], axis=1)
df.info()

In [ ]:
# La misma jugadora aparece en los datos como ganadora y como perdedora.
# Separamos y renombramos para poder unificar y calcular medias globales.

df_winner = df_stats[['winner_name', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn',
       'w_1stWon', 'w_2ndWon', 'w_bpSaved', 'w_bpFaced']].rename(columns={
    'winner_name': 'jugadora', 'w_ace': 'ace', 'w_df': 'df', 'w_svpt': 'svpt', 'w_1stIn': '1stIn',
    'w_1stWon': '1stWon', 'w_2ndWon': '2ndWon', 'w_bpSaved': 'bpSaved', 'w_bpFaced': 'bpFaced'
})

df_loser = df_stats[['loser_name', 'l_ace', 'l_df', 'l_svpt', 'l_1stIn',
                     'l_1stWon', 'l_2ndWon', 'l_bpSaved', 'l_bpFaced']].rename(columns={
    'loser_name': 'jugadora', 'l_ace': 'ace', 'l_df': 'df', 'l_svpt': 'svpt', 'l_1stIn': '1stIn',
    'l_1stWon': '1stWon', 'l_2ndWon': '2ndWon', 'l_bpSaved': 'bpSaved', 'l_bpFaced': 'bpFaced'
})

In [ ]:
from difflib import get_close_matches

def nombre_stats_a_wta(nombre_completo, candidatos_wta):
    """Convierte formato 'Caroline Garcia' → 'Garcia C.' para hacer match con el dataset WTA."""
    if pd.isna(nombre_completo):
        return None
    partes = nombre_completo.strip().split(' ')
    inicial = partes[0][0]
    apellido = ' '.join(partes[1:])
    nombre_convertido = f"{apellido} {inicial}."
    
    if nombre_convertido in candidatos_wta:
        return nombre_convertido
    
    matches = get_close_matches(nombre_convertido, candidatos_wta, n=1, cutoff=0.6)
    return matches[0] if matches else None

candidatos_wta = list(set(wta['Player_1'].unique()) | set(wta['Player_2'].unique()))

In [ ]:
# Unir partidos como ganadora y como perdedora
df_all = pd.concat([df_winner, df_loser], ignore_index=True)

# Incluir % de break points salvados como feature adicional
df_all['bpsaved_per'] = df_all['bpSaved'] / df_all['bpFaced']

# Normalizar nombres al formato del dataset WTA (proceso aproximado por inconsistencias de nomenclatura)
df_all['jugadora_wta'] = df_all['jugadora'].apply(lambda x: nombre_stats_a_wta(x, candidatos_wta))

# Media de estadísticos por jugadora
df_cluster = df_all.drop(columns=['jugadora_wta']).groupby('jugadora').mean()

# Añadir el nombre en formato WTA
mapping = df_all[['jugadora', 'jugadora_wta']].drop_duplicates().set_index('jugadora')
df_cluster['nombre_wta'] = mapping['jugadora_wta']

df_cluster.info()

In [ ]:
cols_numericas = ['ace', 'df', 'svpt', '1stIn', '1stWon', '2ndWon', 'bpSaved', 'bpFaced', 'bpsaved_per']

# Comprobar si hay jugadoras con nulos parciales (algunas columnas sí, otras no)
parciales = df_cluster[df_cluster[cols_numericas].isna().any(axis=1) & ~df_cluster[cols_numericas].isna().all(axis=1)]
print(f'Jugadoras con nulos parciales: {len(parciales)}')

# Eliminar las que tienen TODAS las columnas numéricas nulas (sin datos en absoluto)
df_kmeans = df_cluster.dropna(subset=cols_numericas, how='all')
df_kmeans.info()

In [ ]:
# Caso especial: jugadoras con datos en todo excepto bpsaved_per
# Esto indica que nunca tuvieron break points en contra → se imputa 1.0 (salvaron el 100%)
df_kmeans[df_kmeans['bpsaved_per'].isna() & df_kmeans[['ace', 'df', 'svpt', '1stIn', '1stWon', '2ndWon',
                                                          'bpSaved', 'bpFaced']].notna().all(axis=1)]

In [ ]:
df_kmeans['bpsaved_per'] = df_kmeans['bpsaved_per'].fillna(1.0)

### Método del codo

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

scaler = StandardScaler()
X = df_kmeans[cols_numericas]
X_scaled = scaler.fit_transform(X)

# Método del codo — por lógica deportiva esperaríamos 3-4 clusters (agresivas, defensivas, mixtas...)
inertias = []
k_range = range(2, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.plot(k_range, inertias, marker='o')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inercia')
plt.title('Método del codo')
plt.show()

# No hay un punto de inflexión claro. Se prueba con silhouette score.

### Silhouette score

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []
k_range = range(2, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(k_range, silhouette_scores, marker='o')
plt.title('Coeficiente de silueta')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Silhouette score')
plt.grid(True)
plt.show()

# El máximo está en K=2, pero el score es moderado — no hay una clusterización fuerte.

### Visualización PCA con K=2

In [ ]:
from sklearn.decomposition import PCA

kmeans_2 = KMeans(n_clusters=2, random_state=42)
labels = kmeans_2.fit_predict(X_scaled)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
for cluster in [0, 1]:
    mask = labels == cluster
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {cluster}', alpha=0.7)

plt.title('Clusters de saque (PCA)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend()
plt.grid(True)
plt.show()

print(f"Varianza explicada total: {sum(pca.explained_variance_ratio_)*100:.1f}%")

### Conclusión del clustering

PC1 (46.8%) separa los dos clusters: el Cluster 1 se agrupa a la izquierda y el Cluster 0 a la derecha, lo que indica que PC1 captura la diferencia principal entre los dos perfiles de saque. PC2 (19.4%) apenas discrimina entre grupos.

Hay una zona central con mucho solapamiento, consistente con los scores de silueta moderados. Los clusters existen, pero no son grupos nítidos: son más bien dos extremos de un continuo. **La variable de clustering se descartó como feature del modelo por no aportar separación suficiente.**

Referencia de columnas del dataset de estadísticas:
- `w_ace` / `l_ace`: número de aces de la ganadora/perdedora
- `w_df` / `l_df`: dobles faltas
- `w_svpt` / `l_svpt`: puntos de servicio totales
- `w_1stIn` / `l_1stIn`: primeros saques dentro
- `w_1stWon` / `l_1stWon`: puntos ganados con primer saque
- `w_2ndWon` / `l_2ndWon`: puntos ganados con segundo saque
- `w_bpSaved` / `l_bpSaved`: break points salvados
- `w_bpFaced` / `l_bpFaced`: break points en contra

## 7. Análisis de variables del dataset de features

Correlaciones y distribuciones sobre `historico_partidos` para validar las features construidas y detectar posibles redundancias.

In [ ]:
import seaborn as sns

# Pairplot — visión general de relaciones entre variables
sns.pairplot(data=historico_partidos)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 16))
sns.heatmap(
    historico_partidos.corr(numeric_only=True),
    annot=True,
    fmt='.2f',
    cmap='coolwarm'
)
plt.title('Matriz de correlación — historico_partidos')
plt.tight_layout()
plt.show()

## 8. Features evaluadas y descartadas

Durante el proceso se probaron las siguientes features que finalmente no se incluyeron en el modelo final:

- **`inactividad_p1` / `inactividad_p2`**: días desde el último partido. Se evaluó pero no mejoró el rendimiento del modelo y añadía complejidad de cálculo.
- **`tendencia_ranking`**: diferencia de ranking entre hace 6 meses y ahora. Correlacionada con `rank_diff` y con datos nulos frecuentes.
- **`tournament_type`**: se probó incluirla como feature categórica pero el modelo daba mejor resultado sin ella (ver notebook 02).
- **Clustering de saque**: descartado por los motivos explicados en la sección 6.